# Set Up Data for Sentiment UDF 
**IMPORTANT:** Run this notebook to set up the data needed for the notebook *Python User-Defined Function (UDF) to Detect Sentiment*

This notebook sets up data that will be used in the following notebook, which labels customer comments with detected sentiment scores, and integrates sentiment scores with structured data to produce useful analytic reports.

---
### The structured and semi-structured data
The example includes tables recording customer purchases from vending machines. These tables are created:
- **PRODUCT**: Products vended, including Product ID and Name
- **VENDING_MACHINE**: Machines vending products, including VM ID and Location
- **VENDING_TXN**: Vending transactions, including TXN ID, VM ID, and Product ID

### The unstructured data for labeling
See the discussion in the notebook that follows.

### Steps below:
1. Connect to Snowflake
2. Create and load tables (structured data)
3. Upload unstructured data
4. Upload lexicon file for sentiment analyzer

In [2]:
import snowflake.snowpark
from snowflake.snowpark.functions import *
from snowflake.snowpark.session import Session

import pandas as pd

config_dir = '/home/jovyan/.ssh'
configfile = config_dir + '/sf_config'

### 1. Connect to Snowflake

In [4]:
session = Session.builder.configs({
      "account":   "ES10286-ML_ENTERPRISE",
      "user":      "RKIRK",
      "password":  "8d!upvFs2#BDDB5JQ*7",
      "role":      "RK_SANDPIT_SYSADMIN",
      "warehouse": "DAFT_WH",
      "database":  "RK_SANDPIT",
      "schema":    "DAFT_SCHEMA"
  }).create()

### 2. Create and load tables (structured data)

In [5]:
session.sql("use schema public").collect()

# Table: PRODUCT
session.sql("""create or replace table product (
    prd_id int,
    name string,
    sell_price number(38,2),
    desc string,
    category string,
    buy_price number(38,2))""").collect()
session.file.put("product.csv.gz", "@%product")
session.sql("copy into product purge=true").collect()

# Table: VENDING_MACHINE
session.sql("""create or replace table vending_machine (
    vm_id int,
    loc string,
    cap int,
    status string,
    build_date date)""").collect()
session.file.put("vending_machine.csv.gz", "@%vending_machine")
session.sql("copy into vending_machine purge=true").collect()

# Table: VENDING_TXN
session.sql("""create or replace table vending_txn (
    data object)""").collect()
session.file.put("vending_txn.csv.gz", "@%vending_txn")
session.sql("copy into vending_txn purge=true").collect()

print('Tables PRODUCT, VENDING_MACHINE, and VENDING_TXN created in schema PUBLIC.')

Tables PRODUCT, VENDING_MACHINE, and VENDING_TXN created in schema PUBLIC.


### 3. Upload unstructured data

In [ ]:
# Extract reviews - reviews.zip exists in the same directory as this notebook
import os
os.system('unzip -oq reviews.zip')

# Stage reviews
session.sql("create or replace schema reviewdata").collect()
session.sql("create or replace stage reviewdata.reviews directory = (enable=true)").collect()
session.file.put("reviews/*", "@reviewdata.reviews", auto_compress=False)
session.sql("alter stage reviewdata.reviews refresh").collect()

# Clear extracted reviews
os.system('rm -rf reviews')

0

This creates a schema and a stage on the schema and it loads the review files into the stage;
```sql
SHOW STAGES IN SCHEMA RK_SANDPIT.REVIEWDATA;

LIST @RK_SANDPIT.REVIEWDATA.REVIEWS;
```


### 4. Upload lexicon file for sentiment analyzer

In [8]:
session.sql("create or replace schema util").collect()
session.sql("create or replace stage util.lexicon").collect()
bit_bucket = session.file.put("vader_lexicon.txt", "@util.lexicon", auto_compress=False)

creates another schema + stage and loads the lexicon file into it:
```sql
LIST @RK_SANDPIT.UTIL.LEXICON;
```

#### Claude's explanation of the lexicon file:

 The vader_lexicon.txt is the dictionary that powers the sentiment analysis itself — it's the core data file for the VADER (Valence Aware Dictionary and sEntiment Reasoner) algorithm from the nltk library.

  The file contains thousands of words, phrases, and emoticons, each pre-scored with a valence rating from roughly -4 (most negative) to +4 (most positive). For example, words like "love" or "excellent" have high
  positive scores; words like "disgusting" or "awful" have high negative scores. VADER also includes scores for punctuation patterns, capitalisation, and degree modifiers ("very", "extremely", etc.).

  When the UDF runs inside Snowflake's virtual warehouse, it:
  1. Checks whether /tmp/vader_lexicon.txt already exists on that warehouse node's local filesystem
  2. If not, copies it from the @util.lexicon stage to /tmp
  3. Initialises SentimentIntensityAnalyzer(lexicon_file) pointing at that local file
  
  The reason it needs to be staged and manually copied to /tmp rather than just using nltk.download('vader_lexicon') at runtime is that Snowflake UDFs run in a sandboxed environment with no outbound internet access
   — they can't reach the NLTK download servers. Staging the file in Snowflake and copying it to /tmp is the standard workaround for getting static data files into the UDF runtime.

